# Preparation cost and state windows

This notebook follows a complete research task: bound a preparation's cost, check candidate programs, examine subsystem structure, and pass the saved calculation to an independent reader. The examples use explicit reset and cooling models, followed by a Bell-pair/GHZ comparison.

Start with the [0.3 source quickstart](README.md#quick-start) and [notebook setup](README.md#jupyter-notebook). Select **Python (rcc-refcert)** as the kernel, then run the cells in order. The published 0.2.0 package covers the model and terminal-state tutorials; this notebook uses the cost and window workflows.

You can change the requests below and rerun them. The [workflow guide](docs/COST_AND_WINDOWS.md) describes exact parameters, your own matrices, tensor labels and numerical evidence.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from rcc_refcert.bundle import verify_analysis_bundle
from rcc_refcert.workflows import (
    analyze_cost,
    analyze_windows,
    export_probe,
    load_workflow_template,
    render_workflow_result,
    save_workflow_bundle,
)

## Bound a complete preparation task

The cyclic-reset template fixes a four-qubit model, a complete target in its original basis, and candidate programs. Each physical action costs one atomic slot, including halt. An exact reference-envelope calculation excludes smaller budgets; executing a candidate establishes a feasible upper endpoint when it reaches the target tolerance.

The scan below changes only the allowed trace distance:

| Tolerance | Cost interval | Interpretation |
|---|---|---|
| `1/10` | `[5,5]` | Five slots are necessary and sufficient in this model. |
| `9/20` | `[4,4]` | The four-slot candidate now meets the target. |
| `11/25` | `[4,5]` | The lower bound and best supplied candidate leave a one-slot gap. |

A candidate can miss the target while the analysis completes successfully. To explore a different physical instance, edit the template's `qubits` and `p` parameters before calling `analyze_cost`; use exact strings such as `"19/20"` for probabilities.

In [ ]:
request = load_workflow_template("cyclic-reset")
request["epsilon_values"] = ["1/10", "9/20", "11/25"]
result = analyze_cost(request)
print(render_workflow_result(result))
assert [p["cost"]["interval_slots"] for p in result["epsilon_scan"]] == [
    [5, 5],
    [4, 4],
    [4, 5],
]

## Locate structure with subsystem windows

The first probe compares paired Bell states with a four-qubit GHZ state. Both are globally pure, yet their reduced states organize that structure differently. With `Gamma=4`, the global RCC main term is 2 for both; windows 12, 13 and 34 give `(1,0,1)` for the Bell pairs and `(1/2,1/2,1/2)` for GHZ.

Each window reports entropy, entropy deficit and the deficit divided by the declared reference scale. These are state-side quantities. A preparation-cost bound also uses the model and accuracy information illustrated above.

`export_probe` returns the complete editable request. The probe collection also includes `w2-noise-distribution`, which compares concentrated and distributed phase noise at matched global entropy. Window entropies are numerical, with input and precision diagnostics retained in the result.

In [ ]:
windows = analyze_windows(export_probe("w1-quantum-organization"))
print(render_workflow_result(windows))

## Follow structure along a cooling preparation

The cooling probe refreshes two bath qubits and applies reversible compression to concentrate purity on the target qubit. The request contains both the complete preparation task and windows for its intermediate states, so the profiles follow the actual candidate execution.

At tolerance `1/50`, this task gives a strict `[6,6]` cost interval. Its reference scale is `g_R=log2(3)`, derived from the declared model. Inspect each saved step to see bath refresh remove entropy and correlation, and compression redistribute them. The states are diagonal, so the reported total correlation is classical. Halt adds one charged slot and leaves the state unchanged.

In [ ]:
cooling_request = export_probe("w3-cooling-trajectory")
cooling = analyze_cost(cooling_request)
print(render_workflow_result(cooling))
assert cooling["cost"]["interval_slots"] == [6, 6]
print(render_workflow_result(cooling["windows"]))

## Share a calculation for independent verification

A workflow bundle saves the full effective request and the author's result. Verification reconstructs the model, checks the evidence and reruns the calculation, assigning the recipient's result a new run identifier. The original evidence remains available for comparison.

The cell below writes a bundle in a temporary directory and verifies it independently. `historical_results_supported` should be `True` for this example. To keep a bundle for a collaborator, call `save_workflow_bundle` with a path of your own and share it with the producer version. Cost and window bundles require the 0.3 reader; the [bundle guide](docs/ANALYSIS_BUNDLES.md#complete-cost-and-window-workflows) explains the recipient workflow.

In [ ]:
with TemporaryDirectory() as directory:
    bundle = Path(directory) / "cost.rccbundle"
    save_workflow_bundle(bundle, request, result)
    verification = verify_analysis_bundle(bundle)
    assert verification["historical_results_supported"] is True
    assert verification["fresh_result"]["run_id"] != result["run_id"]
    print(verification["fresh_verification"])